# Milestone 6 — Vitals Analysis

Jupyter analysis over the Iceberg lakehouse (`lake.lakehouse.vitals`, 24
samples at 5-second intervals for a single simulated patient). Reads the table
via Trino and visualizes all six vital channels with pandas + matplotlib.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from trino import dbapi

# The Trino DBAPI needs the catalog/schema wired manually. timezone="UTC"
# matches the lakehouse catalog connection so naive TIMESTAMP literals are not
# shifted to the host's session timezone.
conn = dbapi.connect(
    host="localhost", port=8082, user="notebook",
    timezone="UTC",
)


def load_vitals() -> pd.DataFrame:
    return pd.read_sql(
        """
        SELECT
          event_time,
          heart_rate_bpm,
          temperature_c,
          spo2_pct,
          respiration_rate_bpm,
          systolic_bp_mmhg,
          diastolic_bp_mmhg
        FROM lake.lakehouse.vitals
        ORDER BY event_time
        """,
        conn,
    )


df = load_vitals()
print(f"rows={len(df)}")
df.head()


In [ ]:
channels = {
    "heart_rate_bpm": "Heart rate (bpm)",
    "temperature_c": "Temperature (Celsius)",
    "spo2_pct": "SpO2 (%)",
    "respiration_rate_bpm": "Respiration (rpm)",
    "systolic_bp_mmhg": "Systolic BP (mmHg)",
    "diastolic_bp_mmhg": "Diastolic BP (mmHg)",
}

df[list(channels)].describe().round(2)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for ax, (col, label) in zip(axes.flat, channels.items()):
    ax.plot(df["event_time"], df[col], marker="o", linewidth=2)
    ax.set_title(label)
    ax.grid(alpha=0.3)
axes[0, 0].set_xlabel("event_time")
fig.autofmt_xdate()
fig.suptitle("Vitals over time — lake.lakehouse.vitals", fontsize=14)
fig.tight_layout()
plt.show()


### Headline numbers

- 24 rows spanning 2026-01-01T00:00:05Z..00:01:00Z (5-second cadence).
- Heart rate averages ~72 bpm and SpO2 ~98% — a stable simulated patient.
- The same SQL the notebook runs is reusable verbatim in Grafana, whose Trino
  datasource is provisioned in `infra/grafana/provisioning/datasources/trino.yaml`.
